**read_files**: is a Batch Processing, always read all data (files)

In [0]:
%sql
-- Creates MANAGED Table (Databricks manages Metadata and Data itself)
CREATE OR REPLACE TABLE data_fusion.bronze_delta_tables.historical_users_MANAGED AS
SELECT
        *,
        cast(from_unixtime(user_first_touch_timestamp / 1000000) AS date) converted_date,
        cast(from_unixtime(user_first_touch_timestamp / 1000000) AS timestamp) converted_timestamp,
        _metadata.file_modification_time,
        _metadata.file_name,
        current_timestamp() AS load_time
FROM read_files (
    '/Volumes/data_fusion/source_data/v01/users-historical/',
    format => 'parquet'
)

In [0]:
SELECT
    *
from data_fusion.bronze_delta_tables.historical_users_managed
limit 5

`Python Version`
- rescued data column needs to be added manually

In [0]:
%python
import pyspark.sql.functions as sf
from pyspark.sql.types import *

In [0]:
%python

# reading Data
df = (spark.read
           .format('parquet')
           .option("rescued_Data", "_rescued_data")
           .load('/Volumes/data_fusion/source_data/v01/users-historical/')
      ).withColumn(
            "converted_date", 
            sf.from_unixtime(sf.col("user_first_touch_timestamp")/1000000).cast(DateType())
                  ) \
        .withColumn(
            "converted_timestamp", 
            sf.from_unixtime(sf.col("user_first_touch_timestamp")/1000000).cast(TimestampType())
                  ) \
        .withColumn("file_modification_time", sf.col("_metadata.file_modification_time")) \
        .withColumn("file_name", sf.col("_metadata.file_name")) \
        .withColumn("load_time", sf.current_timestamp())

# writing Data
df.write.mode('overwrite') \
        .option("overwriteSchema", "true") \
        .saveAsTable('data_fusion.bronze_delta_tables.historical_users_MANAGED_python')

In [0]:
%python
display(spark.read.table("data_fusion.bronze_delta_tables.historical_users_managed_python").limit(5))

##### Tables Exploration

In [0]:
DESCRIBE TABLE data_fusion.bronze_delta_tables.historical_users_managed;

In [0]:
DESCRIBE TABLE EXTENDED data_fusion.bronze_delta_tables.historical_users_managed;